In [1]:
import pandas as pd
import numpy as np
import joblib

In [8]:
feature_names = joblib.load("feature_names.pkl")

print("Features expected by the model:")
print(feature_names)

Features expected by the model:
Index(['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide',
       'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index',
       'us_aqi', 'european_aqi', 'year', 'month', 'day', 'day_of_week',
       'us_aqi_lag_1', 'us_aqi_lag_3', 'us_aqi_lag_7', 'us_aqi_roll_3',
       'us_aqi_roll_7', 'us_aqi_roll_std_7'],
      dtype='object')


In [11]:
import joblib
import pandas as pd

# Load trained model
model = joblib.load("karachi_aqi_random_forest.pkl")

# Load feature names
feature_names = joblib.load("feature_names.pkl")

# Load feature configuration
feature_config = joblib.load("feature_config.joblib")

print("Model loaded successfully!")
print("Feature configuration:")
print(feature_config)

Model loaded successfully!
Feature configuration:
{'features': ['pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi', 'year', 'month', 'day', 'day_of_week', 'us_aqi_lag_1', 'us_aqi_lag_3', 'us_aqi_lag_7', 'us_aqi_roll_3', 'us_aqi_roll_7', 'us_aqi_roll_std_7'], 'target': 'target_us_aqi', 'forecast_horizon': '1 day', 'city': 'Karachi', 'country': 'Pakistan'}


In [13]:
import pandas as pd

# Load the historical air quality dataset
df = pd.read_csv(
    r"D:\Projects\Karachi_AQI_Forecasting\data\raw\air_quality\air_quality_historical.csv"
)

# Convert date column
df["date"] = pd.to_datetime(df["date"])

# Sort by date
df = df.sort_values("date").reset_index(drop=True)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nLast 10 records:")
print(df.tail(10))

Dataset loaded successfully!
Dataset shape: (1298, 12)

Last 10 records:
           date       pm10      pm2_5  carbon_monoxide  nitrogen_dioxide  \
1288 2026-02-09  59.991667  32.270833       441.333333         19.387500   
1289 2026-02-10  68.812500  33.658333       545.500000         21.737500   
1290 2026-02-11  60.779167  46.875000       689.500000         29.912500   
1291 2026-02-12  48.233333  39.112500       655.458333         28.604167   
1292 2026-02-13  56.470833  44.216667       905.958333         40.291667   
1293 2026-02-14  32.591667  25.787500       739.291667         28.245833   
1294 2026-02-15  54.070833  33.258333       527.166667         26.354167   
1295 2026-02-16  52.175000  31.879167       458.208333         22.866667   
1296 2026-02-17  51.783333  30.287500       865.541667         27.516667   
1297 2026-02-18  51.070833  31.062500       601.541667         22.658333   

      sulphur_dioxide      ozone  aerosol_optical_depth       dust  uv_index  \
1288      

In [14]:
#Feature Engineering: 

In [15]:
import pandas as pd
import numpy as np

def create_features(data):
    """
    Creates the exact features required by the trained AQI model.
    """

    # Make a copy so the original data is not changed
    data = data.copy()

    # Ensure date is datetime
    data["date"] = pd.to_datetime(data["date"])

    # Sort chronologically
    data = data.sort_values("date").reset_index(drop=True)

    # -----------------------------
    # 1. Time-based features
    # -----------------------------
    data["year"] = data["date"].dt.year
    data["month"] = data["date"].dt.month
    data["day"] = data["date"].dt.day
    data["day_of_week"] = data["date"].dt.dayofweek

    # -----------------------------
    # 2. Lag features
    # -----------------------------
    data["us_aqi_lag_1"] = data["us_aqi"].shift(1)
    data["us_aqi_lag_3"] = data["us_aqi"].shift(3)
    data["us_aqi_lag_7"] = data["us_aqi"].shift(7)

    # -----------------------------
    # 3. Rolling features
    # -----------------------------
    data["us_aqi_roll_3"] = (
        data["us_aqi"].rolling(3).mean()
    )

    data["us_aqi_roll_7"] = (
        data["us_aqi"].rolling(7).mean()
    )

    data["us_aqi_roll_std_7"] = (
        data["us_aqi"].rolling(7).std()
    )

    # -----------------------------
    # Remove rows where lag/rolling
    # features cannot be calculated
    # -----------------------------
    data = data.dropna().reset_index(drop=True)

    return data


# Create all 21 features
processed_df = create_features(df)

print("Raw dataset shape:", df.shape)
print("Processed dataset shape:", processed_df.shape)

print("\nLast record:")
print(processed_df.tail(1))

Raw dataset shape: (1298, 12)
Processed dataset shape: (1287, 22)

Last record:
           date       pm10    pm2_5  carbon_monoxide  nitrogen_dioxide  \
1286 2026-02-18  51.070833  31.0625       601.541667         22.658333   

      sulphur_dioxide      ozone  aerosol_optical_depth       dust  uv_index  \
1286          17.4875  96.458333               0.467083  36.708333  1.291667   

      ...  year  month  day  day_of_week  us_aqi_lag_1  us_aqi_lag_3  \
1286  ...  2026      2   18            2          85.0        87.375   

      us_aqi_lag_7  us_aqi_roll_3  us_aqi_roll_7  us_aqi_roll_std_7  
1286       124.125      91.541667      98.559524          14.342674  

[1 rows x 22 columns]


In [16]:
# Check all columns created by the feature function
print("Columns created by feature engineering:")
print(processed_df.columns.tolist())

print("\nNumber of model features:", len(feature_names))

# Check whether all model features exist
missing_features = [
    feature for feature in feature_names
    if feature not in processed_df.columns
]

extra_features = [
    column for column in processed_df.columns
    if column not in list(feature_names) + ["date"]
]

print("\nMissing features:", missing_features)
print("Extra columns:", extra_features)

Columns created by feature engineering:
['date', 'pm10', 'pm2_5', 'carbon_monoxide', 'nitrogen_dioxide', 'sulphur_dioxide', 'ozone', 'aerosol_optical_depth', 'dust', 'uv_index', 'us_aqi', 'european_aqi', 'year', 'month', 'day', 'day_of_week', 'us_aqi_lag_1', 'us_aqi_lag_3', 'us_aqi_lag_7', 'us_aqi_roll_3', 'us_aqi_roll_7', 'us_aqi_roll_std_7']

Number of model features: 21

Missing features: []
Extra columns: []


In [17]:
#Latest ModelInput: 

In [18]:
# Select the latest row and only the 21 model features
latest_features = processed_df.iloc[[-1]][list(feature_names)]

print("Latest feature date:", processed_df.iloc[-1]["date"])

print("\nLatest model input:")
print(latest_features)

Latest feature date: 2026-02-18 00:00:00

Latest model input:
           pm10    pm2_5  carbon_monoxide  nitrogen_dioxide  sulphur_dioxide  \
1286  51.070833  31.0625       601.541667         22.658333          17.4875   

          ozone  aerosol_optical_depth       dust  uv_index     us_aqi  ...  \
1286  96.458333               0.467083  36.708333  1.291667  91.458333  ...   

      year  month  day  day_of_week  us_aqi_lag_1  us_aqi_lag_3  us_aqi_lag_7  \
1286  2026      2   18            2          85.0        87.375       124.125   

      us_aqi_roll_3  us_aqi_roll_7  us_aqi_roll_std_7  
1286      91.541667      98.559524          14.342674  

[1 rows x 21 columns]


In [19]:
#Verifying Order so the latest Input comes exact same way:

In [21]:
#TEST & Verification:

In [22]:
print("Number of features:", latest_features.shape[1])

print("\nFeature order matches saved configuration:")
print(list(latest_features.columns) == list(feature_names))

# Predict next-day AQI
predicted_next_day_aqi = model.predict(latest_features)[0]

print("\nLatest available date:",
      processed_df.iloc[-1]["date"].date())

print("Predicted next-day AQI:",
      round(predicted_next_day_aqi, 2))

Number of features: 21

Feature order matches saved configuration:
True

Latest available date: 2026-02-18
Predicted next-day AQI: 97.22


In [25]:
# Recreate the training feature matrix from the original raw dataset

training_check_df = df.copy()

# Create time features
training_check_df["year"] = training_check_df["date"].dt.year
training_check_df["month"] = training_check_df["date"].dt.month
training_check_df["day"] = training_check_df["date"].dt.day
training_check_df["day_of_week"] = training_check_df["date"].dt.dayofweek

# Create lag features
training_check_df["us_aqi_lag_1"] = training_check_df["us_aqi"].shift(1)
training_check_df["us_aqi_lag_3"] = training_check_df["us_aqi"].shift(3)
training_check_df["us_aqi_lag_7"] = training_check_df["us_aqi"].shift(7)

# Create rolling features
training_check_df["us_aqi_roll_3"] = (
    training_check_df["us_aqi"].rolling(3).mean()
)

training_check_df["us_aqi_roll_7"] = (
    training_check_df["us_aqi"].rolling(7).mean()
)

training_check_df["us_aqi_roll_std_7"] = (
    training_check_df["us_aqi"].rolling(7).std()
)

# Create the original training target
training_check_df["target_us_aqi"] = (
    training_check_df["us_aqi"].shift(-1)
)

# Remove rows that could not be used for training
training_check_df = training_check_df.dropna().reset_index(drop=True)

# Create the same X used during training
X_recreated = training_check_df[list(feature_names)]

print("Recreated training X shape:", X_recreated.shape)
print("First date:", training_check_df["date"].iloc[0])
print("Last date:", training_check_df["date"].iloc[-1])

Recreated training X shape: (1286, 21)
First date: 2022-08-12 00:00:00
Last date: 2026-02-17 00:00:00


In [26]:
# Select the same date from both pipelines
check_date = pd.Timestamp("2026-02-17")

# Training pipeline features
training_sample = training_check_df.loc[
    training_check_df["date"] == check_date,
    list(feature_names)
].iloc[0]

# Production/reusable pipeline features
production_sample = processed_df.loc[
    processed_df["date"] == check_date,
    list(feature_names)
].iloc[0]

# Compare feature by feature
comparison = pd.DataFrame({
    "training_feature": training_sample,
    "production_feature": production_sample
})

comparison["difference"] = (
    comparison["training_feature"]
    - comparison["production_feature"]
)

print(comparison)

print("\nMaximum absolute difference:")
print(comparison["difference"].abs().max())

                       training_feature  production_feature  difference
pm10                          51.783333           51.783333         0.0
pm2_5                         30.287500           30.287500         0.0
carbon_monoxide              865.541667          865.541667         0.0
nitrogen_dioxide              27.516667           27.516667         0.0
sulphur_dioxide               20.270833           20.270833         0.0
ozone                         71.541667           71.541667         0.0
aerosol_optical_depth          0.448750            0.448750         0.0
dust                          35.000000           35.000000         0.0
uv_index                       1.264583            1.264583         0.0
us_aqi                        85.000000           85.000000         0.0
european_aqi                  62.958333           62.958333         0.0
year                        2026.000000         2026.000000         0.0
month                          2.000000            2.000000     

In [27]:
#Matched Both Training and the production phase like they are working in the similar way basically the training was that one I extracted the model made that feature the roll and the lags, didn't the production work I created a symbol pipeline that when the data comes it should break the data in the similar way create the similar lags & everything and I was just matching that that both things are working the similar way and I got that maximum absolute zero means that both work exactly the same so now it's ready to move on.

In [28]:
#NOW Connect to API (OPEN_METRO) FOR LATEST AIR QUALITY DATA:

In [29]:
import requests
import pandas as pd

# Karachi coordinates
LATITUDE = 24.8607
LONGITUDE = 67.0011

# Exact variables needed by our trained model
hourly_variables = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "aerosol_optical_depth",
    "dust",
    "uv_index",
    "us_aqi",
    "european_aqi"
]

url = "https://air-quality-api.open-meteo.com/v1/air-quality"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "hourly": ",".join(hourly_variables),
    "timezone": "Asia/Karachi",
    "domains": "cams_global",
    "past_days": 7,
    "forecast_days": 1
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("API request successful!")
print("Returned keys:", data.keys())

# Convert hourly API response to DataFrame
hourly_df = pd.DataFrame(data["hourly"])

hourly_df["time"] = pd.to_datetime(hourly_df["time"])

print("\nHourly data shape:", hourly_df.shape)
print("\nFirst 5 rows:")
print(hourly_df.head())

print("\nLast 5 rows:")
print(hourly_df.tail())

API request successful!
Returned keys: dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

Hourly data shape: (192, 12)

First 5 rows:
                 time  pm10  pm2_5  carbon_monoxide  nitrogen_dioxide  \
0 2026-08-28 00:00:00  34.0   16.0            193.0              10.2   
1 2026-08-28 01:00:00  35.9   16.8            154.0               8.5   
2 2026-08-28 02:00:00  29.0   14.2            126.0               7.1   
3 2026-08-28 03:00:00  26.3   12.9            112.0               6.1   
4 2026-08-28 04:00:00  25.6   12.4            108.0               5.4   

   sulphur_dioxide  ozone  aerosol_optical_depth  dust  uv_index  us_aqi  \
0              5.1   43.0                   0.47  23.0       0.0      76   
1              4.7   44.0                   0.47  21.0       0.0      75   
2              4.4   45.0                   0.47  20.0       0.0      75   
3              4.

In [30]:
#Model trained on daily data and the API Gives in Hourly so convert first to daily:

In [31]:
# Create a date-only column
hourly_df["date"] = hourly_df["time"].dt.normalize()

# Columns to aggregate
air_quality_columns = [
    "pm10",
    "pm2_5",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "aerosol_optical_depth",
    "dust",
    "uv_index",
    "us_aqi",
    "european_aqi"
]

# Calculate daily averages
live_daily_df = (
    hourly_df
    .groupby("date")[air_quality_columns]
    .mean()
    .reset_index()
)

print("Live daily data shape:", live_daily_df.shape)

print("\nLive daily records:")
print(live_daily_df)

Live daily data shape: (8, 12)

Live daily records:
        date       pm10      pm2_5  carbon_monoxide  nitrogen_dioxide  \
0 2026-08-28  33.945833  16.891667       194.458333          7.754167   
1 2026-08-29  32.029167  16.358333       189.125000          7.379167   
2 2026-08-30  28.437500  14.316667       153.541667          5.816667   
3 2026-08-31  27.141667  13.787500       189.583333          8.700000   
4 2026-09-01  37.125000  17.145833       212.500000         11.020833   
5 2026-09-02  39.712500  17.441667       229.125000         12.466667   
6 2026-09-03  39.779167  17.137500       212.958333         10.870833   
7 2026-09-04  36.608333  16.870833       200.083333          9.062500   

   sulphur_dioxide      ozone  aerosol_optical_depth       dust  uv_index  \
0         4.750000  52.541667               0.415000  21.833333  1.762500   
1         4.108333  48.583333               0.417500  19.083333  1.560417   
2         3.912500  47.333333               0.461667  16.54

In [32]:
# Count how many hourly records are available for each day
hours_per_day = (
    hourly_df
    .groupby(hourly_df["time"].dt.date)
    .size()
)

print("Hourly records per day:")
print(hours_per_day)

Hourly records per day:
time
2026-08-28    24
2026-08-29    24
2026-08-30    24
2026-08-31    24
2026-09-01    24
2026-09-02    24
2026-09-03    24
2026-09-04    24
dtype: int64


In [33]:
# Create model features from the latest 8 days
live_processed_df = create_features(live_daily_df)

print("Live processed data shape:", live_processed_df.shape)

print("\nProcessed live records:")
print(
    live_processed_df[
        ["date", "us_aqi", "us_aqi_lag_1",
         "us_aqi_lag_3", "us_aqi_lag_7",
         "us_aqi_roll_3", "us_aqi_roll_7",
         "us_aqi_roll_std_7"]
    ]
)

Live processed data shape: (1, 22)

Processed live records:
        date  us_aqi  us_aqi_lag_1  us_aqi_lag_3  us_aqi_lag_7  us_aqi_roll_3  \
0 2026-09-04  66.375     65.166667        61.125     69.291667      66.027778   

   us_aqi_roll_7  us_aqi_roll_std_7  
0         63.625           2.737133  


In [35]:
# Select the latest row in the exact feature order
live_features = live_processed_df.iloc[[-1]][list(feature_names)]

# Predict next-day AQI
live_prediction = model.predict(live_features)[0]

print("Latest live data date:",
      live_processed_df.iloc[-1]["date"].date())

print("Predicted next-day AQI:",
      round(live_prediction, 2))

Latest live data date: 2026-09-04
Predicted next-day AQI: 61.39
